# Notebook 2 — Convergence, trade-offs, and "what should I use when?"

This notebook compares a few **small but representative** adaptation strategies on the **same dataset** and **same pretrained checkpoint**.

Recommended comparison set:

- **Linear probing**
- **BitFit**
- **LoRA**
- **Prompt tuning**

Why these four?

- they are conceptually distinct
- they are easy to explain
- they cover a useful range of cost vs flexibility
- they already produce actionable rules of thumb

This notebook is deliberately minimal. The goal is not leaderboard performance.  
The goal is to help participants answer:

> "Given a dataset and a checkpoint, which PEFT family should I try first?"

In [ ]:
# Optional install cell
# !pip install -q transformers datasets peft accelerate evaluate scikit-learn matplotlib pandas

In [ ]:
import os
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    default_data_collator,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    PromptTuningConfig,
    TaskType,
    get_peft_model,
)

torch.manual_seed(7)
np.random.seed(7)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1) Pick one checkpoint and one dataset

For a workshop, it is often better to use something that runs in minutes rather than hours.

Suggested defaults:

- checkpoint: `distilbert-base-uncased`
- dataset: `ag_news`
- small subset: enough to show trends without long runtimes

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "ag_news"

N_TRAIN = 2000
N_VAL = 1000
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-4

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
raw = load_dataset(DATASET_NAME)

train_ds = raw["train"].shuffle(seed=7).select(range(N_TRAIN))
val_ds = raw["test"].shuffle(seed=7).select(range(N_VAL))

label_names = raw["train"].features["label"].names
num_labels = len(label_names)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

len(train_loader), len(val_loader), label_names

## 2) Helpers

We use one shared train / eval loop for all methods.

In [ ]:
def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total


@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()
    losses = []
    preds = []
    refs = []

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        losses.append(out.loss.item())
        preds.extend(out.logits.argmax(dim=-1).detach().cpu().tolist())
        refs.extend(batch["labels"].detach().cpu().tolist() if "labels" in batch else batch["label"].detach().cpu().tolist())

    return {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(refs, preds),
    }


def standardize_batch(batch):
    batch = dict(batch)
    if "label" in batch and "labels" not in batch:
        batch["labels"] = batch.pop("label")
    return batch


def train_one_method(model, train_loader, val_loader, epochs=3, lr=2e-4, weight_decay=0.01):
    model.to(device)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps,
    )

    history = []
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_losses = []

        for batch in train_loader:
            batch = standardize_batch(batch)
            batch = {k: v.to(device) for k, v in batch.items()}

            out = model(**batch)
            loss = out.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            epoch_losses.append(loss.item())

        val_metrics = evaluate_model(model, [
            {("labels" if k == "label" else k): v for k, v in b.items()} for b in val_loader
        ])

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)),
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "elapsed_sec": time.time() - start,
        })

    return pd.DataFrame(history)

## 3) Build the methods

### Linear probe
Freeze the encoder and train only the classification head.

### BitFit
Train biases (and classification head) only.

### LoRA
Insert low-rank updates into attention projections.

### Prompt tuning
Learn virtual prompt tokens while keeping the base model frozen.

In [ ]:
def build_linear_probe():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    for name, p in model.named_parameters():
        p.requires_grad = ("classifier" in name or "pre_classifier" in name)
    return model


def build_bitfit():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    for name, p in model.named_parameters():
        p.requires_grad = ("bias" in name or "classifier" in name or "pre_classifier" in name)
    return model


def build_lora():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["q_lin", "v_lin"],
        bias="none",
    )
    model = get_peft_model(model, config)
    return model


def build_prompt_tuning():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    config = PromptTuningConfig(
        task_type=TaskType.SEQ_CLS,
        num_virtual_tokens=20,
        prompt_tuning_init="RANDOM",
    )
    model = get_peft_model(model, config)
    return model


builders = {
    "linear_probe": build_linear_probe,
    "bitfit": build_bitfit,
    "lora": build_lora,
    "prompt_tuning": build_prompt_tuning,
}

In [ ]:
summary_rows = []
for name, builder in builders.items():
    model = builder()
    trainable, total, pct = count_params(model)
    summary_rows.append({
        "method": name,
        "trainable_params": trainable,
        "total_params": total,
        "trainable_%": pct,
    })

param_df = pd.DataFrame(summary_rows).sort_values("trainable_params")
param_df

## 4) Run the comparison

On CPU this may take a bit.  
For a live workshop, using GPU or reducing `N_TRAIN`, `N_VAL`, or `EPOCHS` is enough to keep the exercise lightweight.

In [ ]:
all_histories = []
final_rows = []

for name, builder in builders.items():
    print(f"\n=== Training {name} ===")
    model = builder()
    hist = train_one_method(model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
    hist["method"] = name
    all_histories.append(hist)

    last = hist.iloc[-1].to_dict()
    trainable, total, pct = count_params(model)
    final_rows.append({
        "method": name,
        "trainable_params": trainable,
        "trainable_%": pct,
        "final_train_loss": last["train_loss"],
        "final_val_loss": last["val_loss"],
        "final_val_acc": last["val_acc"],
        "elapsed_sec": last["elapsed_sec"],
    })

history_df = pd.concat(all_histories, ignore_index=True)
results_df = pd.DataFrame(final_rows).sort_values("final_val_acc", ascending=False)
results_df

In [ ]:
plt.figure(figsize=(8, 4))
for method, g in history_df.groupby("method"):
    plt.plot(g["epoch"], g["val_acc"], marker="o", label=method)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Convergence comparison")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
for method, g in history_df.groupby("method"):
    plt.plot(g["epoch"], g["train_loss"], marker="o", label=method)
plt.xlabel("Epoch")
plt.ylabel("Train loss")
plt.title("Optimization behavior")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(results_df["trainable_params"], results_df["final_val_acc"])
for _, row in results_df.iterrows():
    plt.annotate(row["method"], (row["trainable_params"], row["final_val_acc"]))
plt.xscale("log")
plt.xlabel("Trainable parameters (log scale)")
plt.ylabel("Final validation accuracy")
plt.title("Accuracy vs adaptation budget")
plt.show()

## 5) Turn curves into decisions

The point of this notebook is to move from "which method is cooler?" to a more practical question:

> "Given **my** dataset and **my** checkpoint, which family should I start with?"

Below is a simple decision template you can adapt after each run.

In [ ]:
def recommend_from_results(results_df):
    best = results_df.sort_values("final_val_acc", ascending=False).iloc[0]
    cheapest_good = results_df.sort_values(["final_val_acc", "trainable_params"], ascending=[False, True]).iloc[0]

    notes = []
    notes.append(f"Best accuracy in this run: {best['method']} ({best['final_val_acc']:.3f})")
    notes.append(
        f"Best accuracy-per-small-budget candidate: {cheapest_good['method']} "
        f"with {int(cheapest_good['trainable_params']):,} trainable params"
    )

    return notes

for line in recommend_from_results(results_df):
    print("-", line)

## 6) A compact "what to use when" cheat-sheet

Use the empirical results above together with the following working heuristics.

### Start with linear probing when...
- you need a **strong sanity-check baseline**
- the task is close to what the checkpoint already knows
- you want to know whether the representation is already separable

### Try BitFit when...
- you want an **ultra-cheap** update
- the domain shift seems mild
- you want a baseline slightly more flexible than linear probing

### Try LoRA when...
- you want a very strong **default PEFT baseline**
- you expect the model needs to actually change how it computes
- you want a good accuracy / efficiency compromise

### Try prompt tuning when...
- you need to keep the base checkpoint fully frozen
- you want many task-specific states on top of one shared model
- your deployment pattern favors storing prompts rather than weight deltas

---

## Suggested workshop framing

A nice workshop message is:

1. **Linear probe first** → "Are frozen features already enough?"
2. **BitFit next** → "Can tiny parameter nudges solve it?"
3. **LoRA next** → "Do I need a stronger weight-space update?"
4. **Prompt tuning in parallel** → "Can I steer the model from the input side instead?"

That framing gives participants a practical ladder rather than a long disconnected list of PEFT names.

## 7) Optional extensions for your HAICON workshop

Good follow-up variants if you want one extra exercise:

- add **IA3** as another multiplicative / scale-style method
- compare **LoRA vs AdaLoRA** under a fixed parameter budget
- add a second dataset with a larger distribution shift and see whether the ranking changes
- swap `distilbert-base-uncased` for a checkpoint participants already use in practice
- record not just accuracy, but also:
  - trainable parameter count
  - peak memory
  - wall-clock per epoch
  - checkpoint size

That gives a much better answer to "**what should I use when?**" than accuracy alone.